# 2. Walk-forward modeling: 3-class, 60-minute horizon

The model predicts DOWN, FLAT, or UP for the close 60 minutes after the latest completed 15-minute bar. FLAT means the move is smaller than 0.20% of the current price, which prevents tiny price noise from becoming a forced direction.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "s3_to_dataframe.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from s3_to_dataframe import load_data_from_s3, prepare_dataframe
from feature_engineering import create_features
from modeling import train_direction_model, walk_forward_validate, save_model, predict_latest

TIMEFRAME = "15min"
HORIZON_MINUTES = 60
HORIZON_BARS = HORIZON_MINUTES // 15
FLAT_THRESHOLD = 0.002

raw_df = prepare_dataframe(load_data_from_s3())
print(f"Raw S3 candle rows: {len(raw_df):,}")
feature_df = create_features(
    raw_df, timeframe=TIMEFRAME, horizon_bars=HORIZON_BARS,
    flat_threshold=FLAT_THRESHOLD
)
print(f"15-minute model rows after feature warmup and target horizon: {len(feature_df):,}")
print(feature_df["target"].value_counts().rename({-1: "DOWN", 0: "FLAT", 1: "UP"}))

Raw S3 candle rows: 35,004
15-minute model rows after feature warmup and target horizon: 34,094
target
FLAT    16553
UP       8799
DOWN     8742
Name: count, dtype: int64


Walk-forward validation uses expanding training windows. Every test block occurs after its training block, so no future rows are used to train an earlier prediction.

In [2]:
walk_forward = walk_forward_validate(feature_df, folds=5)
fold_table = pd.DataFrame([
    {"fold": item["fold"], "train_rows": item["train_rows"],
     "test_rows": item["test_rows"], "accuracy": item["accuracy"]}
    for item in walk_forward["folds"]
])
display(fold_table)
print(f"Largest-class baseline: {feature_df['target'].value_counts(normalize=True).max():.3f}")
print(f"Mean walk-forward accuracy: {walk_forward['mean_accuracy']:.3f}")

,fold,train_rows,test_rows,accuracy
0,1,17047,3409,0.515107
1,2,20456,3409,0.519801
2,3,23865,3409,0.473159
3,4,27274,3409,0.647111
4,5,30683,3409,0.557642


Largest-class baseline: 0.486
Mean walk-forward accuracy: 0.543


In [3]:
model, holdout = train_direction_model(feature_df)
print(f"Final chronological holdout accuracy: {holdout['accuracy']:.3f}")
display(pd.DataFrame(holdout["confusion_matrix"], index=["DOWN", "FLAT", "UP"], columns=["pred DOWN", "pred FLAT", "pred UP"]))
display(pd.DataFrame(holdout["classification_report"]).T)
save_model(model, {"holdout": holdout, "walk_forward": walk_forward, "configuration": {"timeframe": TIMEFRAME, "horizon_minutes": HORIZON_MINUTES, "flat_threshold": FLAT_THRESHOLD}}, ROOT / "models" / "btcusdt_direction_15m_60m.joblib")

Final chronological holdout accuracy: 0.602


,pred DOWN,pred FLAT,pred UP
DOWN,120,1072,116
FLAT,120,3819,174
UP,148,1084,166


,precision,recall,f1-score,support
-1,0.309278,0.091743,0.141509,1308.000000
0,0.639163,0.928519,0.757137,4113.000000
1,0.364035,0.118741,0.179072,1398.000000
accuracy,0.601994,0.601994,0.601994,0.601994
macro avg,0.437492,0.379668,0.359240,6819.000000
weighted avg,0.519480,0.601994,0.520537,6819.000000


A live prediction needs historical context. The API call therefore retrieves a rolling window of completed 15-minute candles, not one row. The feature pipeline calculates moving averages, RSI, volatility, and other trailing features from that window, then only the latest feature row is passed to the model.

In [4]:
from data_fetcher import get_binance_data

live_candles = prepare_dataframe(pd.DataFrame(get_binance_data()))
prediction = predict_latest(model, live_candles, timeframe=TIMEFRAME)
print(f"Completed bar: {prediction['bar_time_utc']}")
print(f"Current close: ${prediction['close']:,.2f}")
print(f"Predicted direction in {HORIZON_MINUTES} minutes: {prediction['prediction']}")
print(prediction["probabilities"])

Completed bar: 2026-09-26T23:45:00+00:00
Current close: $84,451.58
Predicted direction in 60 minutes: FLAT
{'DOWN': 0.1900166255176292, 'FLAT': 0.6793658906175031, 'UP': 0.1306174838648677}
